# 04 — Hybrid Retrieval

**Pipeline position:** After embedding (03), before reranking (05).

**Purpose:** Combine BM25 sparse retrieval with Chroma dense retrieval using
Reciprocal Rank Fusion (RRF), then merge and deduplicate via parent-child lookup.

**Learning objectives:**
- Understand BM25 vs dense retrieval trade-offs
- Implement RRF fusion: `score = sum(1/(k + rank))`
- Use `merge_docs()` to fetch parent documents from MongoDB
- Compare retrieval quality across methods

## Imports

In [ ]:
import sys
sys.path.insert(0, '..')

from src import constant
from src.retriever.bm25_retriever import BM25Retriever
from src.retriever.chroma_retriever import ChromaRetriever
from src.pipeline import rrf_fuse
from src.utils import merge_docs

## Load retrievers

BM25 loads from pickle; Chroma connects to the persistent vector store.

In [ ]:
# BM25
bm25 = BM25Retriever(docs=None, retrieve=False)

# Dense — needs an encoder function
from src.pipeline import RAGPipeline
encode_fn = RAGPipeline._load_default_encoder()
dense = ChromaRetriever(encode_query_fn=encode_fn)

## BM25 retrieval example

BM25 excels at exact keyword matching — good for specific terms like gene names or cell types.

In [ ]:
query = 'What is the role of MHC class II in antigen presentation?'

bm25_results = bm25.retrieve_topk(query, topk=5)
print(f'BM25 results for: "{query}"\n')
for i, doc in enumerate(bm25_results, 1):
    src = doc.metadata.get('source_file', '?')[:30]
    pg = doc.metadata.get('page', '?')
    print(f'[{i}] (p{pg}, {src}) {doc.page_content[:100]}...\n')

## Dense retrieval example

Dense retrieval captures semantic similarity — finds passages that mean the same thing even with different vocabulary.

In [ ]:
dense_results = dense.retrieve_topk(query, topk=5)
print(f'Dense results for: "{query}"\n')
for i, doc in enumerate(dense_results, 1):
    src = doc.metadata.get('source_file', '?')[:30]
    pg = doc.metadata.get('page', '?')
    dist = doc.metadata.get('_chroma_distance', 0)
    print(f'[{i}] (p{pg}, dist={dist:.4f}) {doc.page_content[:100]}...\n')

## RRF Fusion

Reciprocal Rank Fusion combines both lists:
```
score(doc) = bm25_weight * 1/(k + rank_bm25) + dense_weight * 1/(k + rank_dense)
```
where k=60 (smoothing constant). Documents appearing in both lists get boosted.

In [ ]:
fused = rrf_fuse(bm25_results, dense_results, k=60, bm25_weight=0.5, dense_weight=0.5)
print(f'Fused results ({len(fused)} unique docs):\n')
for i, doc in enumerate(fused[:5], 1):
    uid = doc.metadata.get('unique_id', '?')[:12]
    print(f'[{i}] (uid={uid}) {doc.page_content[:100]}...\n')

## merge_docs() — Parent document lookup

Child chunks are precise for matching but too small for LLM context.
`merge_docs()` looks up the parent chunk in MongoDB for richer context.

In [ ]:
merged = merge_docs(bm25_results, dense_results)
print(f'Merged parent docs: {len(merged)}\n')
for i, doc in enumerate(merged[:3], 1):
    has_parent = 'parent_id' in doc.metadata
    print(f'[{i}] (has_parent={has_parent}, len={len(doc.page_content)} chars)')
    print(f'    {doc.page_content[:150]}...\n')

## Compare methods

Side-by-side unique document IDs from each retrieval method.

In [ ]:
def uids(docs):
    return set(d.metadata.get('unique_id', '') for d in docs if d.metadata.get('unique_id'))

bm25_uids = uids(bm25_results)
dense_uids = uids(dense_results)
overlap = bm25_uids & dense_uids

print(f'BM25 unique docs:  {len(bm25_uids)}')
print(f'Dense unique docs: {len(dense_uids)}')
print(f'Overlap:           {len(overlap)}')
print(f'Union:             {len(bm25_uids | dense_uids)}')
print(f'\nOverlap ratio: {len(overlap)/max(len(bm25_uids | dense_uids),1):.1%}')

## Summary

**Key takeaways:**
- BM25 is strong for exact keyword matches; dense retrieval captures semantic similarity
- RRF fusion combines both strengths without needing score normalization
- `merge_docs()` upgrades child chunks to parent chunks for better LLM context

**Next:** `05_reranker.ipynb` — Cross-encoder reranking with BGE-reranker-v2-m3.